# Miami smelter: inspect the raw seven-band archive
Run from the project directory with the project's `.venv` Python kernel. Data cover 2016-01-01 through 2022-11-30. These are raw packed arrays; physical-value decoding and quality cleaning are later steps. The AOI is a provisional OSM industrial polygon, not an operator-confirmed boundary.

In [ ]:
import sys
import importlib.util
print('Notebook Python:', sys.executable)
required = ['rioxarray', 'xarray', 'zarr', 'psutil', 'pandas', 'numpy', 'matplotlib']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Missing packages: ' + ', '.join(missing) + '. Select the EI-Nightlights (.venv) kernel, restart it, and Run All.')
from pathlib import Path
import json
import pandas as pd
from viirs_zarr import open_product, BANDS
root = Path('data/miami_zarr')
a1 = open_product(root / 'VNP46A1.zarr')
a2 = open_product(root / 'VNP46A2.zarr')
a1

In [ ]:
rows = []
for ds in (a1, a2):
    rows.append({'product': ds.attrs['product'], 'calendar_days': ds.sizes['time'], 'acquired_days': int(ds.source_available.sum()), 'unlisted_days': int((ds.source_available == 0).sum()), 'crs': str(ds.rio.crs)})
pd.DataFrame(rows)

In [ ]:
sample = a2.sel(time=slice('2016-01-01', '2016-01-06')).load()
print(sample[BANDS['VNP46A2']])
print('Pixel centres:', sample.x.values, sample.y.values)
print('AOI intersection mask:', sample.aoi_intersects_pixel.values)
print('Per-date native band metadata:')
json.loads(sample.scene_band_metadata.values[0])

In [ ]:
# A raw pixel view only. No scaling, fill removal or cloud screening.
sample['DNB_BRDF-Corrected_NTL'].isel(time=0).plot(x='x', y='y', cmap='magma')

In [ ]:
a1.close()
a2.close()